**SOP CREATION**

In [1]:
!pip install groq pandas -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.7 MB/s eta 0:00:00


In [3]:
import os, json, re
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd

# ==== PROJECT DIRECTORY ====
PROJECT_DIR = "/content/Genai Project"
os.makedirs(PROJECT_DIR, exist_ok=True)

# ==== FORCE CORRECT PATHS (AS YOU SPECIFIED) ====
TRAIN_FILE = "/content/train.jsonl"
VAL_FILE = "/content/val.jsonl"

# ==== DEBUG: SHOW FILES IN /content ====
print("Files in /content directory:")
print(os.listdir("/content"))

# ==== VALIDATION ====
if not Path(TRAIN_FILE).exists():
    raise FileNotFoundError(f"train.jsonl NOT FOUND at {TRAIN_FILE}")

if not Path(VAL_FILE).exists():
    raise FileNotFoundError(f"val.jsonl NOT FOUND at {VAL_FILE}")

print("Using TRAIN_FILE:", TRAIN_FILE)
print("Using VAL_FILE:", VAL_FILE)

# ==== READ JSONL ====
def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

rows = read_jsonl(TRAIN_FILE) + read_jsonl(VAL_FILE)
print("Loaded records:", len(rows))

# ==== HELPER ====
def top_items(counter_obj, k=5):
    return [x for x, _ in counter_obj.most_common(k)]

# ==== PROFILE STRUCTURE ====
profiles = defaultdict(lambda: {
    "count": 0,
    "severity_counts": Counter(),
    "primary_dept_counts": Counter(),
    "hospital_dept_counts": Counter(),
    "first_responder_counts": Counter(),
    "resolution_owner_counts": Counter(),
    "sla_hours_counts": Counter(),
    "action_on_report_counts": Counter(),
    "escalation_chain_samples": [],
    "visual_indicators": Counter(),
    "subtypes": Counter(),
    "voice_samples": [],
    "caption_samples": [],
    "description_samples": [],
    "reported_internally_counts": Counter(),
    "physical_harm_counts": Counter(),
    "evidence_clarity_counts": Counter(),
    "consistency_score_stats": [],
})

# ==== MAIN LOOP ====
for r in rows:
    cat = str(r.get("category", "")).strip()
    if not cat:
        continue
    p = profiles[cat]
    p["count"] += 1

    sev = str(r.get("severity", "")).lower().strip()
    if sev:
        p["severity_counts"][sev] += 1

    primary_dept = ((r.get("routing", {}) or {}).get("primary_department") or "").strip()
    if primary_dept:
        p["primary_dept_counts"][primary_dept] += 1

    hosp_dept = ((r.get("hospital_details", {}) or {}).get("department") or "").strip()
    if hosp_dept:
        p["hospital_dept_counts"][hosp_dept] += 1

    first_responder = ((r.get("routing", {}) or {}).get("first_responder") or "").strip()
    if first_responder:
        p["first_responder_counts"][first_responder] += 1

    resolution_owner = ((r.get("routing", {}) or {}).get("resolution_owner") or "").strip()
    if resolution_owner:
        p["resolution_owner_counts"][resolution_owner] += 1

    sla = (r.get("routing", {}) or {}).get("sla_hours")
    if sla is not None:
        p["sla_hours_counts"][str(sla)] += 1

    action = ((r.get("routing", {}) or {}).get("action_on_report") or "").strip()
    if action:
        p["action_on_report_counts"][action] += 1

    esc_chain = (r.get("routing", {}) or {}).get("escalation_chain") or []
    if isinstance(esc_chain, list) and esc_chain:
        p["escalation_chain_samples"].append(esc_chain)

    taxonomy = r.get("taxonomy", {}) or {}
    for v in (taxonomy.get("visual_indicators") or []):
        p["visual_indicators"][str(v).strip()] += 1
    for s in (taxonomy.get("subtypes") or []):
        p["subtypes"][str(s).strip()] += 1

    incident = r.get("incident_details", {}) or {}
    p["reported_internally_counts"][str(incident.get("reported_internally"))] += 1
    p["physical_harm_counts"][str(incident.get("physical_harm"))] += 1

    evidence = r.get("evidence", {}) or {}
    p["evidence_clarity_counts"][str(evidence.get("evidence_clarity"))] += 1

    v = str(r.get("voice_text", "")).strip()
    c = str((r.get("input", {}) or {}).get("image_caption") or r.get("refined_caption", "")).strip()
    d = str(r.get("complaint_description", "")).strip()

    if v:
        p["voice_samples"].append(v)
    if c:
        p["caption_samples"].append(c)
    if d:
        p["description_samples"].append(d)

    validation = r.get("validation_layer", {}) or {}
    sc = validation.get("consistency_score")
    if isinstance(sc, (int, float)):
        p["consistency_score_stats"].append(float(sc))

# ==== FINAL PROFILE BUILD ====
final_profiles = {}
for cat, p in profiles.items():
    final_profiles[cat] = {
        "record_count": p["count"],
        "severity_distribution": dict(p["severity_counts"]),
        "top_primary_departments": top_items(p["primary_dept_counts"], 5),
        "top_hospital_departments": top_items(p["hospital_dept_counts"], 5),
        "top_first_responders": top_items(p["first_responder_counts"], 5),
        "top_resolution_owners": top_items(p["resolution_owner_counts"], 5),
        "top_sla_hours": top_items(p["sla_hours_counts"], 5),
        "top_action_on_report": top_items(p["action_on_report_counts"], 3),
        "sample_escalation_chain": p["escalation_chain_samples"][0] if p["escalation_chain_samples"] else [],
        "top_visual_indicators": top_items(p["visual_indicators"], 10),
        "top_subtypes": top_items(p["subtypes"], 10),
        "reported_internally_distribution": dict(p["reported_internally_counts"]),
        "physical_harm_distribution": dict(p["physical_harm_counts"]),
        "evidence_clarity_distribution": dict(p["evidence_clarity_counts"]),
        "consistency_score_avg": round(
            sum(p["consistency_score_stats"]) / len(p["consistency_score_stats"]), 4
        ) if p["consistency_score_stats"] else None,
        "voice_examples": p["voice_samples"][:3],
        "caption_examples": p["caption_samples"][:3],
        "description_examples": p["description_samples"][:3],
    }

print("Categories profiled:", len(final_profiles))
print("Category names:", list(final_profiles.keys()))

Files in /content directory:
['.config', 'test.jsonl', 'train.jsonl', 'Genai Project', 'val.jsonl', 'sample_data']
Using TRAIN_FILE: /content/train.jsonl
Using VAL_FILE: /content/val.jsonl
Loaded records: 185
Categories profiled: 10
Category names: ['Broken Hospital Bed', 'Crowded Hospital Waiting Room', 'Dirty Hospital Bathroom', 'Empty / Unstaffed Nursing Station', 'Overflowing Hospital Trash (Outside)', 'Rats / Rodent Infestation', 'Torn Hospital Privacy Curtain', 'Unappetizing Hospital Food Tray', 'Unhygienic / Contaminated Hospital Food', 'Water Puddle on Hospital Floor']


In [9]:
# Cell 2.5 - Load Groq key (Colab-safe)
import os

token = None

# Try Colab Secrets first
try:
    from google.colab import userdata
    token = userdata.get("GROQ_API_KEY") or userdata.get("GROQ_API")
except Exception:
    pass

# Fallback to already-set env vars
token = token or os.getenv("GROQ_API_KEY") or os.getenv("GROQ_API")

if not token:
    raise EnvironmentError(
        "Groq key missing. In Colab: add secret named GROQ_API_KEY (or GROQ_API), then rerun."
    )

os.environ["GROQ_API_KEY"] = token
print("GROQ_API_KEY loaded:", bool(os.environ.get("GROQ_API_KEY")))


GROQ_API_KEY loaded: True


In [10]:
from groq import Groq
client = Groq(api_key=os.environ["GROQ_API_KEY"])
print("Groq client ready.")


Groq client ready.


In [11]:
import json, re
from pathlib import Path

SOP_DIR = Path(PROJECT_DIR) / "sop_docs"
SOP_DIR.mkdir(parents=True, exist_ok=True)

def slugify(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return re.sub(r"_+", "_", text).strip("_")

SYSTEM_PROMPT = """You are generating hospital triage SOP documents.
Rules:
1) Use only provided profile data.
2) Produce practical, safety-oriented SOP in Markdown.
3) Include these exact sections:
   - SOP Title
   - Scope
   - Trigger Indicators
   - Severity Policy
   - Department Routing
   - First Response Actions
   - Escalation Chain
   - SLA Targets
   - Human Review Triggers
   - Audit Fields
4) Keep it operational, not theoretical.
5) Do not invent unknown regulations.
"""

def build_user_prompt(category, profile):
    return f"""
Category: {category}

Profile JSON:
{json.dumps(profile, indent=2, ensure_ascii=False)}

Generate one SOP document for this category.
"""

def fallback_sop(category, profile):
    sev_dist = profile.get("severity_distribution", {})
    common_sev = next(iter(sorted(sev_dist, key=lambda x: sev_dist[x], reverse=True)), "medium")
    depts = profile.get("top_primary_departments", [])
    first_resp = profile.get("top_first_responders", [])
    owners = profile.get("top_resolution_owners", [])
    sla = profile.get("top_sla_hours", [])
    triggers = profile.get("top_visual_indicators", [])[:6]
    action = profile.get("top_action_on_report", ["Follow standard incident response protocol."])[0]
    esc = profile.get("sample_escalation_chain", [])

    return f"""# SOP Title
SOP for {category}

## Scope
This SOP applies to incidents classified as **{category}**.

## Trigger Indicators
{chr(10).join(f"- {x}" for x in triggers) if triggers else "- Visual/voice evidence indicating this complaint category."}

## Severity Policy
- Default severity baseline: **{common_sev}**
- Escalate to **high/critical** if patient safety risk or repeated unresolved incidents are detected.

## Department Routing
- Primary department(s): {", ".join(depts) if depts else "To be determined by triage lead"}
- Resolution owner(s): {", ".join(owners) if owners else "To be assigned"}

## First Response Actions
- First responder role(s): {", ".join(first_resp) if first_resp else "Duty Nurse / Ward In-Charge"}
- Immediate action: {action}

## Escalation Chain
{chr(10).join(f"- {x}" for x in esc) if esc else "- Ward In-Charge -> Department Supervisor -> Hospital Administrator"}

## SLA Targets
- Expected SLA hours: {", ".join(sla) if sla else "4-8 hours (category dependent)"}

## Human Review Triggers
- Low confidence structured output
- Cross-modal mismatch (caption vs voice)
- Ambiguous department ownership
- Safety-critical signal with incomplete evidence

## Audit Fields
- incident_id
- timestamp
- category
- severity_before_rules
- severity_after_rules
- primary_department
- escalation_owner
- action_taken
- needs_human_review
"""

def generate_sop(category, profile):
    user_prompt = build_user_prompt(category, profile)
    try:
        resp = client.chat.completions.create(
            model=SELECTED_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.2,
            max_tokens=1200,
        )
        text = (resp.choices[0].message.content or "").strip()
        if len(text) < 200:
            return fallback_sop(category, profile), "fallback_short_output"
        return text, "llm_ok"
    except Exception as e:
        return fallback_sop(category, profile), f"fallback_error:{str(e)[:120]}"


In [13]:
# Fix missing vars before Cell 5
from datetime import datetime
from pathlib import Path
import os

SELECTED_MODEL = globals().get("SELECTED_MODEL") or globals().get("MODEL_NAME") or "llama-3.3-70b-versatile"

if "TRAIN_FILE" not in globals():
    TRAIN_FILE = "/content/train.jsonl" if Path("/content/train.jsonl").exists() else "/train.jsonl"
if "VAL_FILE" not in globals():
    VAL_FILE = "/content/val.jsonl" if Path("/content/val.jsonl").exists() else "/val.jsonl"
if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = "/content/sop_outputs"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

print("SELECTED_MODEL:", SELECTED_MODEL)


SELECTED_MODEL: llama-3.3-70b-versatile


In [14]:
import pandas as pd
import json
from datetime import datetime

records = []
for category, profile in final_profiles.items():
    sop_text, status = generate_sop(category, profile)
    fname = f"sop_{slugify(category)}.md"
    fpath = SOP_DIR / fname
    fpath.write_text(sop_text, encoding="utf-8")

    records.append({
        "category": category,
        "file_name": fname,
        "status": status,
        "record_count": profile.get("record_count", 0),
    })
    print(f"[{status}] {category} -> {fname}")

index_payload = {
    "generated_at_utc": datetime.utcnow().isoformat(),
    "model_used": SELECTED_MODEL,
    "source_train_file": TRAIN_FILE,
    "source_val_file": VAL_FILE,
    "sop_dir": str(SOP_DIR),
    "items": records,
}
(SOP_DIR / "sop_index.json").write_text(json.dumps(index_payload, indent=2), encoding="utf-8")

df_report = pd.DataFrame(records).sort_values(["category"])
df_report.to_csv(SOP_DIR / "sop_generation_report.csv", index=False)

print("\nSaved:")
print("-", SOP_DIR / "sop_index.json")
print("-", SOP_DIR / "sop_generation_report.csv")
print("-", f"{len(records)} SOP markdown files")


[llm_ok] Broken Hospital Bed -> sop_broken_hospital_bed.md
[llm_ok] Crowded Hospital Waiting Room -> sop_crowded_hospital_waiting_room.md
[llm_ok] Dirty Hospital Bathroom -> sop_dirty_hospital_bathroom.md
[llm_ok] Empty / Unstaffed Nursing Station -> sop_empty_unstaffed_nursing_station.md
[llm_ok] Overflowing Hospital Trash (Outside) -> sop_overflowing_hospital_trash_outside.md
[llm_ok] Rats / Rodent Infestation -> sop_rats_rodent_infestation.md
[llm_ok] Torn Hospital Privacy Curtain -> sop_torn_hospital_privacy_curtain.md
[llm_ok] Unappetizing Hospital Food Tray -> sop_unappetizing_hospital_food_tray.md
[llm_ok] Unhygienic / Contaminated Hospital Food -> sop_unhygienic_contaminated_hospital_food.md
[llm_ok] Water Puddle on Hospital Floor -> sop_water_puddle_on_hospital_floor.md

Saved:
- /content/Genai Project/sop_docs/sop_index.json
- /content/Genai Project/sop_docs/sop_generation_report.csv
- 10 SOP markdown files


/tmp/ipykernel_7104/2523662656.py:21: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at_utc": datetime.utcnow().isoformat(),


In [15]:
import zipfile
from pathlib import Path

zip_path = Path(PROJECT_DIR) / "sop_docs_bundle.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fp in SOP_DIR.rglob("*"):
        if fp.is_file():
            zf.write(fp, arcname=str(fp.relative_to(SOP_DIR.parent)))

print("ZIP created at:", zip_path)


ZIP created at: /content/Genai Project/sop_docs_bundle.zip


**ingestion/indexing.**

In [18]:
!pip install -q chromadb sentence-transformers pandas

In [20]:
# Cell 1: SOP + train/val ingestion into Chroma (standalone, no project import)
import json
from pathlib import Path
from datetime import datetime

import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer

# ---- Your exact paths ----
PROJECT_DIR = Path("/content/Genai Project")
SOP_DIR = Path("/content/Genai Project/sop_docs")
TRAIN_FILE = Path("/content/train.jsonl")
VAL_FILE = Path("/content/val.jsonl")

CHROMA_PATH = PROJECT_DIR / "vectordb" / "chromadb"
PROOF_DIR = PROJECT_DIR / "ingestion_proof"
PROOF_DIR.mkdir(parents=True, exist_ok=True)

SOP_COLLECTION = "hospital_sops"
TRAINVAL_COLLECTION = "hospital_trainval"

# ---- Validate ----
if not SOP_DIR.exists():
    raise FileNotFoundError(f"SOP_DIR not found: {SOP_DIR}")
if not TRAIN_FILE.exists():
    raise FileNotFoundError(f"TRAIN_FILE not found: {TRAIN_FILE}")
if not VAL_FILE.exists():
    raise FileNotFoundError(f"VAL_FILE not found: {VAL_FILE}")

print("Using:")
print("SOP_DIR:", SOP_DIR)
print("TRAIN_FILE:", TRAIN_FILE)
print("VAL_FILE:", VAL_FILE)

def norm(s):
    return " ".join((s or "").strip().split())

def chunk_text(text, source, chunk_chars=1800, overlap=200):
    text = norm(text)
    if not text:
        return []
    out = []
    step = max(1, chunk_chars - overlap)
    i, start = 0, 0
    while start < len(text):
        end = min(start + chunk_chars, len(text))
        part = text[start:end].strip()
        if len(part) >= 50:
            out.append({"id": f"{Path(source).stem}_chunk_{i}", "source": source, "text": part})
            i += 1
        if end == len(text):
            break
        start += step
    return out

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path=str(CHROMA_PATH))

# Recreate collections
for cname in [SOP_COLLECTION, TRAINVAL_COLLECTION]:
    try:
        client.delete_collection(cname)
    except Exception:
        pass
sop_coll = client.create_collection(name=SOP_COLLECTION, metadata={"hnsw:space": "cosine"})
tv_coll = client.create_collection(name=TRAINVAL_COLLECTION, metadata={"hnsw:space": "cosine"})

# --- SOP indexing ---
sop_files = sorted([p for p in SOP_DIR.iterdir() if p.is_file() and p.suffix.lower() in {".md", ".txt"}])
sop_chunks = []
for fp in sop_files:
    sop_chunks.extend(chunk_text(fp.read_text(encoding="utf-8", errors="ignore"), fp.name))

sop_emb = embedder.encode([c["text"] for c in sop_chunks], normalize_embeddings=True).astype(np.float32).tolist()
batch = 128
for i in range(0, len(sop_chunks), batch):
    part = sop_chunks[i:i+batch]
    sop_coll.add(
        ids=[x["id"] for x in part],
        embeddings=sop_emb[i:i+batch],
        documents=[x["text"] for x in part],
        metadatas=[{"source": x["source"]} for x in part],
    )

# --- train+val indexing ---
rows = read_jsonl(TRAIN_FILE) + read_jsonl(VAL_FILE)
tv_chunks = []
for r in rows:
    rid = str(r.get("image_id", "unknown"))
    caption = (r.get("input", {}) or {}).get("image_caption") or r.get("refined_caption", "")
    voice = r.get("voice_text", "")
    category = r.get("category", "")
    severity = str(r.get("severity", "")).lower()
    dept = (r.get("routing", {}) or {}).get("primary_department") or r.get("department", "")
    desc = r.get("complaint_description", "")
    txt = (
        f"Case ID: {rid}\n"
        f"Observed caption: {caption}\n"
        f"Voice complaint: {voice}\n"
        f"Known category: {category}\n"
        f"Known severity: {severity}\n"
        f"Known department: {dept}\n"
        f"Known complaint_description: {desc}\n"
    )
    tv_chunks.extend(chunk_text(txt, f"trainval_{rid}.txt"))

tv_emb = embedder.encode([c["text"] for c in tv_chunks], normalize_embeddings=True).astype(np.float32).tolist()
for i in range(0, len(tv_chunks), batch):
    part = tv_chunks[i:i+batch]
    tv_coll.add(
        ids=[x["id"] for x in part],
        embeddings=tv_emb[i:i+batch],
        documents=[x["text"] for x in part],
        metadatas=[{"source": x["source"]} for x in part],
    )

result = {
    "timestamp_utc": datetime.utcnow().isoformat(),
    "chroma_path": str(CHROMA_PATH),
    "sop_collection": {"name": SOP_COLLECTION, "source_docs": len(sop_files), "chunks_indexed": len(sop_chunks)},
    "trainval_collection": {"name": TRAINVAL_COLLECTION, "rows_used": len(rows), "chunks_indexed": len(tv_chunks)},
}
(PROOF_DIR / "ingestion_result.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
print(json.dumps(result, indent=2))


Using:
SOP_DIR: /content/Genai Project/sop_docs
TRAIN_FILE: /content/train.jsonl
VAL_FILE: /content/val.jsonl


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

{
  "timestamp_utc": "2026-04-25T23:02:36.886679",
  "chroma_path": "/content/Genai Project/vectordb/chromadb",
  "sop_collection": {
    "name": "hospital_sops",
    "source_docs": 10,
    "chunks_indexed": 22
  },
  "trainval_collection": {
    "name": "hospital_trainval",
    "rows_used": 185,
    "chunks_indexed": 185
  }
}


/tmp/ipykernel_7104/1777545941.py:128: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat(),


In [21]:
# Cell 2: Proof (counts + retrieval sample)
import json
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer

PROJECT_DIR = Path("/content/Genai Project")
CHROMA_PATH = PROJECT_DIR / "vectordb" / "chromadb"
PROOF_DIR = PROJECT_DIR / "ingestion_proof"

client = chromadb.PersistentClient(path=str(CHROMA_PATH))
sop = client.get_collection("hospital_sops")
tv = client.get_collection("hospital_trainval")

counts = {"hospital_sops_count": sop.count(), "hospital_trainval_count": tv.count()}
(PROOF_DIR / "collection_counts.json").write_text(json.dumps(counts, indent=2), encoding="utf-8")
print("Counts:", counts)

query = "Rats near hospital kitchen, contamination risk, urgent cleaning required."
emb = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2").encode(query, normalize_embeddings=True).tolist()

sop_out = sop.query(query_embeddings=[emb], n_results=3, include=["documents","metadatas","distances"])
tv_out = tv.query(query_embeddings=[emb], n_results=3, include=["documents","metadatas","distances"])

proof = {
    "query": query,
    "sop_top_ids": sop_out.get("ids", [[]])[0],
    "trainval_top_ids": tv_out.get("ids", [[]])[0],
}
(PROOF_DIR / "retrieval_proof.json").write_text(json.dumps(proof, indent=2), encoding="utf-8")
print(json.dumps(proof, indent=2))
print("Proof folder:", str(PROOF_DIR))


Counts: {'hospital_sops_count': 22, 'hospital_trainval_count': 185}


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{
  "query": "Rats near hospital kitchen, contamination risk, urgent cleaning required.",
  "sop_top_ids": [
    "sop_rats_rodent_infestation_chunk_1",
    "sop_rats_rodent_infestation_chunk_0",
    "sop_dirty_hospital_bathroom_chunk_1"
  ],
  "trainval_top_ids": [
    "trainval_rats_11_chunk_0",
    "trainval_rats_15_chunk_0",
    "trainval_rats_14_chunk_0"
  ]
}
Proof folder: /content/Genai Project/ingestion_proof


In [22]:
# Cell 3: Zip proof folder for submission/download
import zipfile
from pathlib import Path
from google.colab import files

proof_dir = Path("/content/Genai Project/ingestion_proof")
zip_path = Path("/content/Genai Project/ingestion_proof.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in proof_dir.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(proof_dir.parent))

print("Created:", zip_path)
files.download(str(zip_path))


Created: /content/Genai Project/ingestion_proof.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
# Zip + download full Chroma vectordb folder (Colab)
import zipfile
from pathlib import Path
from google.colab import files

src_dir = Path("/content/Genai Project/vectordb")
zip_path = Path("/content/Genai Project/vectordb_chromadb.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in src_dir.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(src_dir.parent))

print("Created:", zip_path)
files.download(str(zip_path))


Created: /content/Genai Project/vectordb_chromadb.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>